In [25]:
from google.cloud import bigquery
from google.oauth2 import service_account

#### Create client and link dataset Google BigQuery

In [26]:
def create_biguery_client ():
    # Set up authentication using a service account
    credentials = service_account.Credentials.from_service_account_file('bq_key.json')
    # Create a BigQuery client using the credentials
    client = bigquery.Client(credentials=credentials, project=credentials.project_id)
    print(f"The Client:\n ${client}\n was successfully created.")
    return client


In [27]:
client = create_biguery_client()

The Client:
 $<google.cloud.bigquery.client.Client object at 0x126007b10>
 was successfully created.


#### Construct a reference to a BigQuery dataset

In [28]:
project_id = "bigquery-public-data"
dataset_id = "san_francisco"

def create_dataset(client):
    # Construct a reference to the "stackoverflow" dataset
    dataset_ref = client.dataset(f"{dataset_id}", project=f"{project_id}")

    # API request - fetch the dataset
    dataset = client.get_dataset(dataset_ref)
    print(f"The dataset:\n ${dataset}\n has been successfully created ")
    return dataset_ref, dataset

In [29]:
dataset, dataset_ref = create_dataset(client)

The dataset:
 $Dataset(DatasetReference('bigquery-public-data', 'san_francisco'))
 has been successfully created 


#### Explore the data

##### List the available tables

In [30]:
# Get a list of available tables
tables = list(client.list_tables(dataset))
list_of_tables = [table.table_id for table in tables]

# Print the list of tables (uncomment print type for debug)
# print(list_of_tables)
# print(type(list_of_tables))
for table in list_of_tables:
    print(table)


311_service_requests
bikeshare_stations
bikeshare_status
bikeshare_trips
film_locations
sffd_service_calls
sfpd_incidents
street_trees


#### Construct a reference to the relevant table
##### bikeshare_trips

In [31]:
table_id = "bikeshare_trips"
def create_bigquery_table(client, dataset_ref):
    # Construct a reference to the table
    answers_table_ref = dataset_ref.table(f"{table_id}")

    # API request - fetch the table
    answers_table = client.get_table(answers_table_ref)

    # Preview the first five lines of the table
    print(client.list_rows(answers_table, max_results=5).to_dataframe())

In [32]:
create_bigquery_table(client,dataset_ref)

   trip_id  duration_sec                start_date  \
0  1235850          1540 2016-06-11 08:19:00+00:00   
1  1219337          6324 2016-05-29 12:49:00+00:00   
2   793762        115572 2015-06-04 09:22:00+00:00   
3   453845         54120 2014-09-15 16:53:00+00:00   
4  1245113          5018 2016-06-17 20:08:00+00:00   

                  start_station_name  start_station_id  \
0  San Jose Diridon Caltrain Station                 2   
1  San Jose Diridon Caltrain Station                 2   
2  San Jose Diridon Caltrain Station                 2   
3  San Jose Diridon Caltrain Station                 2   
4  San Jose Diridon Caltrain Station                 2   

                   end_date                   end_station_name  \
0 2016-06-11 08:45:00+00:00  San Jose Diridon Caltrain Station   
1 2016-05-29 14:34:00+00:00  San Jose Diridon Caltrain Station   
2 2015-06-05 17:28:00+00:00  San Jose Diridon Caltrain Station   
3 2014-09-16 07:55:00+00:00  San Jose Diridon Caltrain Station

#### Construct SQL queries 

Dry run to validate query and estimate bytes

In [33]:
# Query to count the (cumulative) number of trips per day
num_trips_query = f"""
                  WITH trips_by_day AS
                  (
                  SELECT DATE(start_date) AS trip_date,
                    COUNT(*) as num_trips
                  FROM `{project_id}.{dataset_id}.{table_id}`
                  WHERE EXTRACT(YEAR FROM start_date) = 2015
                  GROUP BY trip_date
                  )
                  SELECT *,
                    SUM(num_trips)
                        OVER (
                            ORDER BY trip_date
                            ROWS BETWEEN UNBOUNDED PRECEDING 
                            AND CURRENT ROW
                        ) AS cumulative_trips
                        FROM trips_by_day
"""
# Create a "QueryJobConfig" object to set the "dry_run"
dry_run_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)

# API request - dry run query to estimate cost of query
dry_run_query_job = client.query(num_trips_query, job_config=dry_run_config)

# Print estimated number of bytes processed by the query
print(f"This query will process {dry_run_query_job.total_bytes_processed} bytes.")

This query will process 7869184 bytes.


Run the query with safe_config

In [34]:
import warnings
# Suppress bigquery storage module warning
# Could also install 'pip install google-cloud-bigquery-storage'
warnings.filterwarnings("ignore", message=".*BigQuery Storage module not found.*")

# Set up the query (a real service would have good error handling for 
# queries that scan too much data)
safe_config = bigquery.QueryJobConfig(maximum_bytes_billed=10**10)      
my_query_job = client.query(num_trips_query, job_config=safe_config)

# API request - run the query, and return a pandas DataFrame
results = my_query_job.to_dataframe()

# print(results)
results.head()


,trip_date,num_trips,cumulative_trips
0,2015-01-01,181,181
1,2015-01-02,428,609
2,2015-01-03,283,892
3,2015-01-04,206,1098
4,2015-01-05,1186,2284


Dry run to validate query and estimate bytes

In [35]:
# Query to track beginning and ending stations on October 25, 2015, for each bike
start_end_query = f"""
    SELECT bike_number,
    TIME(start_date) AS trip_time,
    FIRST_VALUE(start_station_id)
        OVER (
            PARTITION BY bike_number
            ORDER BY start_date
            ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
        ) AS first_station_id,
    LAST_VALUE(end_station_id)
        OVER (
            PARTITION BY bike_number
            ORDER BY start_date
            ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
        ) AS last_station_id,
        start_station_id,
        end_station_id
        FROM `{project_id}.{dataset_id}.{table_id}`
        WHERE DATE(start_date) = '2015-10-25'
"""
# Create a "QueryJobConfig" object to set the "dry_run"
dry_run_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)

# API request - dry run query to estimate cost of query
dry_run_query_job = client.query(num_trips_query, job_config=dry_run_config)

# Print estimated number of bytes processed by the query
print(f"This query will process {dry_run_query_job.total_bytes_processed} bytes.")

This query will process 7869184 bytes.


Run the query with safe_config

In [36]:
import warnings
# Suppress bigquery storage module warning
# Could also install 'pip install google-cloud-bigquery-storage'
warnings.filterwarnings("ignore", message=".*BigQuery Storage module not found.*")

# Set up the query (a real service would have good error handling for 
# queries that scan too much data)
safe_config = bigquery.QueryJobConfig(maximum_bytes_billed=10**10)      
my_query_job = client.query(start_end_query, job_config=safe_config)

# API request - run the query, and return a pandas DataFrame
results = my_query_job.to_dataframe()

# print(results)
results.head()


,bike_number,trip_time,first_station_id,last_station_id,start_station_id,end_station_id
0,22,13:25:00,2,16,2,16
1,25,11:43:00,77,51,77,60
2,25,12:14:00,77,51,60,51
3,29,14:59:00,46,74,46,60
4,29,21:23:00,46,74,60,74
